# Studying nonconvex constellations $(458,3240)$ via $\Delta\Phi(x,p)$

Note: This notebook extends the analysis of nonconvex constellations to $J=458$.  This notebook is an adaptation of the notebook <i>23_nonconvex_prep_Engelsma459</i> for $(458,3240)$.

The graph of $\Delta \Phi(x,p)$ for the Engelsma counterexamples $J=458, \; |s|=3240$ is in the 25th block below.

We apply $\Delta\Phi(x,p)$ and its visualizations to the study of Engelsma's $(458,3240)$
examples of nonconvex constellations.  These are relatively long admissible constellations of low span
such that
$$ \pi(|s|) < {\rm length}(s).$$
These examples $s$ show that if the $k$-tuple conjecture is true, then the convexity conjecture
for $\pi(x)$ is false.  For such an $s$ we have
$$ \pi(\gamma_0+|s|) > \pi(\gamma_0) + \pi(|s|).$$

We analyze counterexamples $s$ with $(J,|s|)=(458,3240)$.
We start with a driving term for $s$ in ${\mathcal G}(11^\#)$ with initial generators 
${\gamma=107,\; 109,\; 1271,\; 1273}$
and trace their evolution through subsequent stages of the sieve.

Guided by step R2 of the recursion $R: {\mathcal G}(p_{k-1}^\#) \longrightarrow {\mathcal G}(p_k^\#)$,
we use primorial coordinates or the primorial expansion for $\gamma_0(p_k)$ to track the incidences of $s$.
$$ \gamma_0(p_k) = \gamma_0 + m_1 \cdot 11^\# + m_2 \cdot 13^\# + \cdots + m_k \cdot p_{k-1}^\#$$
or 
$$ \gamma_0(p_k) = \gamma_0 + 11^\# (m_1 + 13(m_2 + 17(m_3 + \cdots +  p_{k-1}^\# \cdot m_k ))\cdots)$$
Each coefficient $m_i$ lies in the range $0 \le m_i < p_i$ and indicates which copy of ${\mathcal G}(p_{i-1}^\#)$ this image of $s$ lies in
under step R2 for ${\mathcal G}(p_{i-1}^\#)\longrightarrow {\mathcal G}(p_i^\#)$.

In [1]:
import pandas as pd
import numpy as np
import array
import itertools
from sympy import mod_inverse
import random

import matplotlib.pyplot as plt
from ipywidgets import interact
import ipywidgets as widgets
from IPython.display import display
plt.rcParams['figure.dpi'] = 300
plt.ion

import gc
import psutil
import sys
import csv
import pickle

In [2]:
# set up the array of small primes.  Start with primes19
primes19=np.load('primes19.npy')

In [3]:
primes19[0:20]

array([ 23,  29,  31,  37,  41,  43,  47,  53,  59,  61,  67,  71,  73,
        79,  83,  89,  97, 101, 103, 107])

In [4]:
# PRIMES:  The array smallp runs through primes from 2 up to 9699691
# 
smallp = np.concatenate(([2,3,5,7,11,13,17,19],primes19))
smallp[0:52], smallp[-5:]

(array([  2,   3,   5,   7,  11,  13,  17,  19,  23,  29,  31,  37,  41,
         43,  47,  53,  59,  61,  67,  71,  73,  79,  83,  89,  97, 101,
        103, 107, 109, 113, 127, 131, 137, 139, 149, 151, 157, 163, 167,
        173, 179, 181, 191, 193, 197, 199, 211, 223, 227, 229, 233, 239]),
 array([9699647, 9699649, 9699653, 9699667, 9699691]))

In [5]:
# Comparing other arrays of primes available to us
# primesE9 = np.load('primesE9.npy')
# primesE9[-10:]

In [6]:
# primes46E9 has max prime 4623102269
# primes46E9 = np.load('primesE9_46.npy')
# primes46E9[-100:]

This notebook automates the analysis of Engelsma $(J,|s|)$-counterexamples to the convexity conjecture 
for constellations among primes.
We use $\Delta \Phi$ to visualize the constellation.  
The array DelPhi contains the lower points on the vertical segments for $\Delta \Phi(x,p)$.  The upper points on these segments are (DelPhi+1)

In [7]:
# block to check the available system memory
gc.collect()
memory = psutil.virtual_memory()
available_memory = memory.available
del memory
print(f"Available memory: {available_memory / (1024 ** 2):.2f} MB")

Available memory: 5012.58 MB


## Engelsma counterexamples to the convexity conjecture
Engelsma et al. have identified several examples of admissible constellations for which their lengths $J$ exceed the number of primes under their span
$$ \pi(|s|) < J$$
The shortest counterexamples $s$ have length $J=458$ and span $|s|=3240$.  These counterexamples can be extended by a single gap $2$ to produce a second admissible counterexample of length $J=459$ and span $|s|=3242$.

These constellations have unique driving terms in the cycle ${\mathcal G}(11^\#)$ with various $\gamma_0$.  
This driving term has a unique image through the next several stages of the sieve.

In [8]:
# This function returns a list of available residues mod inp that begin admissible images of constellation incons
def admissible(inp, incons):
    # calculate list of covered residues mod p by the input constellation
    rez=0
    covered_rez={0}
    i=0
    while (i < len(incons)):
        rez = (rez + incons[i])% inp
        if rez not in covered_rez:
            covered_rez.add(rez)
        i += 1
    # are all residues covered?
    n_available_rez = inp - len(covered_rez)
    i=1
    available_rez = set()
    # each entry in covered_rez corresponds to a starting residue of (inp-rez)mod inp
    while (i < inp):
        test_rez = inp - i
        if test_rez not in covered_rez:
            available_rez.add(i)
        i += 1
    return available_rez

In [9]:
# this function calculates the value of mk such that 
#  0 <= mk < pk  and  mk*pml(p_{k-1}) + r0 = rk mod pk
# reminder for indexing that pk = smallp[k+4], a shift of 4, and p0=11
def primorialm(k,rk,r0):
    pk = smallp[k+4]
    i = 0
    pmlp = 1
    while (i < k+4):
        pmlp = (pmlp * smallp[i]) % pk
        i += 1
    mk = (mod_inverse(pmlp, pk) * (rk-r0) ) % pk
    return mk

In [10]:
# This function returns the generator in G(11#) that begins a driving term for the input constellation
def findgamma11(incons):
    i=1
    gamma0 = 1
    pmlp = 2
    
    while (i <= 4):  # smallp[4]=11
        pk = int(smallp[i])
        rez = admissible(pk, incons)

        if (len(rez) != 1):   # We assume that the generator in G(11#) is unique
            print(f"UNEXPECTED: p {pk} rez {len(rez)} {rez}")

        target_r = int(list(rez)[0])
        r0 = gamma0 % pk
        mk = (mod_inverse(pmlp, pk) * (target_r-r0)) % pk

        gamma0 += mk*pmlp
        pmlp = (pmlp * smallp[i])

        i += 1

    return gamma0        


## We start with (459,3242)-counterexamples
We read in the 58 $(459,3242)$-counterexamples and create the 116 $(458,3240)$-counterexamples by dropping the initial $2$ and 
final $2$ from the longer counterexamples.

In [11]:
# Read in the (459,3242) constellations from text file
Eng459_constellations = np.loadtxt('tuples_460_3242.txt', dtype=int)
Eng459_constellations.shape, Eng459_constellations.shape[0]
# Save as numpy file


((58, 459), 58)

In [12]:
#
Eng459_constellations[0:30,0:12]

array([[ 2,  4, 14,  4,  6,  2, 10,  2,  6,  6, 10,  6],
       [ 2,  4, 14,  4,  6,  2, 10,  2,  6,  6, 10,  6],
       [ 2,  4,  2,  4,  8,  6,  4,  2, 10,  6,  2,  6],
       [ 2,  4,  2,  4,  8,  6,  4,  2, 10,  6,  2,  6],
       [ 2,  4, 14,  4,  6,  2, 10,  2,  6,  6, 10,  6],
       [ 2,  4,  2,  4,  8,  6,  4,  2, 10,  6,  2,  6],
       [ 2,  4,  2,  4,  8,  6,  4,  2, 10,  6,  2,  6],
       [ 2,  4, 14,  4,  6,  2, 10,  2,  6,  6, 10,  6],
       [ 2,  4,  2,  4,  8,  6,  4,  2, 10,  6,  2,  6],
       [ 2,  4,  2,  4,  8,  6,  4,  2, 10,  6,  2,  6],
       [ 2,  4,  2,  4,  8,  6,  4,  2, 10,  6,  2,  6],
       [ 2,  4, 14,  4,  6,  2, 10,  2,  6,  6, 10,  6],
       [ 2,  4, 14,  4,  6,  2, 10,  2,  6,  6, 10,  6],
       [ 2,  4,  2,  4,  8,  6,  4,  2, 10,  6,  2,  6],
       [ 2,  4,  2,  4,  8,  6,  4,  2, 10,  6,  2,  6],
       [ 2,  4,  2,  4,  8,  6,  4,  2, 10,  6,  2,  6],
       [ 2,  4, 14,  4,  6,  2, 10,  2,  6,  6, 10,  6],
       [ 2,  4, 14,  4,  6,  2,

In [13]:
# Extract the (458,3240)-constellations from the (459,3242) data
Eng458_constellations = np.zeros((116,458), dtype=int)
i = 0
while (i < 58):
    Eng458_constellations[i,:] = Eng459_constellations[i,0:458]
    Eng458_constellations[(i+58),:] = Eng459_constellations[i,1:459]
    i += 1
Eng458_constellations[0:10,0:12], Eng458_constellations[58:68,0:12]

(array([[ 2,  4, 14,  4,  6,  2, 10,  2,  6,  6, 10,  6],
        [ 2,  4, 14,  4,  6,  2, 10,  2,  6,  6, 10,  6],
        [ 2,  4,  2,  4,  8,  6,  4,  2, 10,  6,  2,  6],
        [ 2,  4,  2,  4,  8,  6,  4,  2, 10,  6,  2,  6],
        [ 2,  4, 14,  4,  6,  2, 10,  2,  6,  6, 10,  6],
        [ 2,  4,  2,  4,  8,  6,  4,  2, 10,  6,  2,  6],
        [ 2,  4,  2,  4,  8,  6,  4,  2, 10,  6,  2,  6],
        [ 2,  4, 14,  4,  6,  2, 10,  2,  6,  6, 10,  6],
        [ 2,  4,  2,  4,  8,  6,  4,  2, 10,  6,  2,  6],
        [ 2,  4,  2,  4,  8,  6,  4,  2, 10,  6,  2,  6]]),
 array([[ 4, 14,  4,  6,  2, 10,  2,  6,  6, 10,  6,  2],
        [ 4, 14,  4,  6,  2, 10,  2,  6,  6, 10,  6,  2],
        [ 4,  2,  4,  8,  6,  4,  2, 10,  6,  2,  6, 12],
        [ 4,  2,  4,  8,  6,  4,  2, 10,  6,  2,  6, 12],
        [ 4, 14,  4,  6,  2, 10,  2,  6,  6, 10,  6,  2],
        [ 4,  2,  4,  8,  6,  4,  2, 10,  6,  2,  6, 12],
        [ 4,  2,  4,  8,  6,  4,  2, 10,  6,  2,  6, 12],
        [ 4,

In [14]:
Eng458_constellations.shape

(116, 458)

In [15]:
i = 0
while (i < 116):
    const_k = Eng458_constellations[i]
    gamma0 = findgamma11(const_k)
    print(f"{i} {len(const_k)} {np.sum(const_k)} gamma0 {gamma0}")
    i += 1

0 458 3240 gamma0 107
1 458 3240 gamma0 107
2 458 3240 gamma0 1271
3 458 3240 gamma0 1271
4 458 3240 gamma0 107
5 458 3240 gamma0 1271
6 458 3240 gamma0 1271
7 458 3240 gamma0 107
8 458 3240 gamma0 1271
9 458 3240 gamma0 1271
10 458 3240 gamma0 1271
11 458 3240 gamma0 107
12 458 3240 gamma0 107
13 458 3240 gamma0 1271
14 458 3240 gamma0 1271
15 458 3240 gamma0 1271
16 458 3240 gamma0 107
17 458 3240 gamma0 107
18 458 3240 gamma0 1271
19 458 3240 gamma0 107
20 458 3240 gamma0 107
21 458 3240 gamma0 1271
22 458 3240 gamma0 1271
23 458 3240 gamma0 1271
24 458 3240 gamma0 107
25 458 3240 gamma0 107
26 458 3240 gamma0 107
27 458 3240 gamma0 107
28 458 3240 gamma0 1271
29 458 3240 gamma0 107
30 458 3240 gamma0 1271
31 458 3240 gamma0 107
32 458 3240 gamma0 107
33 458 3240 gamma0 1271
34 458 3240 gamma0 1271
35 458 3240 gamma0 1271
36 458 3240 gamma0 107
37 458 3240 gamma0 107
38 458 3240 gamma0 1271
39 458 3240 gamma0 1271
40 458 3240 gamma0 107
41 458 3240 gamma0 1271
42 458 3240 gamma0 127

## Main loop through Engelsma (458,3240)-counterexamples
What follows is the main loop through the 116 counterexamples identified by Thomas Engelsma for the case $(J,|s|)=(458,3240)$.
The first blocks calculate the unique prefix for each counterexample's primorial coordinates 
and sort the counterexamples by their prefixes.

Then for each constellation we search for primorial expansions that have sequences of 0's.

In [16]:
# ================================================================================
# Identify the prefix for the primorial expansion for each (458,3240)-counterexample.
# The prefix is the primorial expansion across the G(p#) for which there is a unique instance
#   for the counterexample.
# ================================================================================
debug_verbose = False
num_cons = Eng458_constellations.shape[0]

Eng458_prefixes = []

# For each constellation --
icons = 0

while (icons < num_cons):
    constellation_k = Eng458_constellations[icons]
    gammam_list = []
    
    # Calculate gamma0 in G(11#)
    gamma0 = findgamma11(constellation_k)

    # Calculate residues across ranges of primes p
    # For each prime >= 11, so index i >= 4, we record the list of admissible residues for gamma_0 mod p[i]
    rezlist = []
    i=4
    while (smallp[i] < 250):
        p = smallp[i]
        rezp = admissible(p,constellation_k)
        rezlist.append(rezp)
        i += 1

    # Summarize num_admissible across the primes p
    num_admissible = [len(rezlist[i]) for i in range(len(rezlist))]
    num_admissible = np.array(num_admissible)

    if debug_verbose:
        print(f"{icons:2d} gamma0 {gamma0:4d} num_adm {num_admissible[0:60]}")
        for element in rezlist:
            print(f" {list(element)[0]}", end=' ')
        print()

    # Calculate primorial coordinates for unique prefix
    ip = 1
    gamma_m = [int(gamma0)]

    while (num_admissible[ip] == 1):
        pk = smallp[ip+4]  # primes are offset 4 in array, p0=11 so we start at pk=13
        
        # calculate the residue r0 mod pk
        r0 = gamma_m[0] % pk
        i = 1
        rpml = 2310 % pk  # 11# mod pk
        while (i < ip):
            r0 = (r0 + gamma_m[i]*rpml) % pk
            rpml = (rpml * smallp[i+4]) % pk
            i += 1

        target_r = list(rezlist[ip])[0]
        mk = primorialm(ip,target_r,r0)
        gamma_m.append(int(mk))
        ip += 1  # next prime

    
    # Report and save this information
    Eng458_prefixes.append(gamma_m)

    print(f"{icons:3d} prefix {Eng458_prefixes[icons]}")
    icons += 1   # next constellation - 


  0 prefix [107, 6, 8, 9, 5, 7, 1, 23, 38, 34, 46, 20, 13, 13, 39, 42, 17, 23, 21, 20, 83, 22, 9, 81, 107, 103, 8, 53]
  1 prefix [107, 6, 8, 9, 5, 7, 1, 23, 38, 34, 46, 20, 13, 13, 39, 42, 17, 23, 21, 20, 49, 64, 50, 10, 29, 28, 27, 38]
  2 prefix [1271, 5, 8, 9, 17, 21, 29, 13, 2, 8, 0, 32, 45, 47, 27, 28, 55, 55, 61, 68, 47, 36, 52, 96, 79, 84, 99, 92]
  3 prefix [1271, 5, 8, 9, 17, 21, 29, 13, 2, 8, 0, 32, 45, 47, 27, 28, 55, 55, 61, 70, 23, 51, 75, 2, 86, 23, 71, 104]
  4 prefix [107, 6, 8, 9, 5, 7, 1, 23, 38, 34, 46, 20, 13, 13, 39, 42, 17, 23, 21, 18, 73, 49, 27, 104, 22, 89, 55, 26]
  5 prefix [1271, 5, 8, 9, 17, 21, 29, 13, 2, 8, 0, 32, 45, 47, 27, 28, 55, 55, 61, 70, 52, 33, 55, 75, 37, 98, 107, 74]
  6 prefix [1271, 5, 8, 9, 17, 21, 29, 13, 2, 8, 0, 32, 45, 47, 27, 28, 55, 55, 61, 68, 76, 18, 32, 62, 31, 46, 9, 63]
  7 prefix [107, 6, 8, 9, 5, 7, 1, 23, 38, 34, 46, 20, 13, 13, 39, 42, 17, 23, 21, 20, 20, 82, 70, 44, 77, 66, 117, 67]
  8 prefix [1271, 5, 8, 9, 17, 21, 29, 13,

In [17]:
Eng_pre_sorted, Eng_consts = zip(*sorted(zip(Eng458_prefixes, Eng458_constellations)))

## NOTE: sorted by prefix of primorial coordinates
At this point the arrays are sorted by their unique prefixes of primorial coordinates with $p_0=11$

In [18]:
i=0
while (i < len(Eng_pre_sorted)):
    print(f"{i:2d} {len(Eng_pre_sorted[i])} m {Eng_pre_sorted[i]}")
    print(f" {Eng_consts[i][0:24]}")
    i += 1

 0 29 m [107, 6, 8, 9, 5, 7, 1, 23, 38, 34, 46, 20, 13, 4, 4, 53, 64, 11, 39, 27, 17, 44, 1, 78, 22, 108, 29, 112, 72]
 [ 2  4 14  4  6  2 10  2  6  6 10  6  2 10  6  2 12 10  2  4 12  2  6  4]
 1 29 m [107, 6, 8, 9, 5, 7, 1, 23, 38, 34, 46, 20, 13, 5, 60, 51, 70, 42, 25, 33, 22, 73, 67, 66, 95, 17, 109, 111, 99]
 [ 2  4 14  4  6  2 10  8  6 10  6  2 10  6  2 12 10  2  4 12  2  6  4  6]
 2 29 m [107, 6, 8, 9, 5, 7, 1, 23, 38, 34, 46, 20, 13, 5, 60, 51, 70, 42, 25, 33, 35, 51, 12, 18, 55, 43, 42, 44, 132]
 [ 2  4 14  4  6  2 10  8  6 10  6  2 10  6  2 12 10  2  4 12  2  6  4  6]
 3 29 m [107, 6, 8, 9, 5, 7, 1, 23, 38, 34, 46, 20, 13, 5, 60, 51, 70, 42, 25, 33, 37, 1, 4, 60, 40, 21, 71, 114, 10]
 [ 2  4 14  4  6  2 10  8  6 10  6  2 10  6  2 12 10  2  4 12  2  6  4  6]
 4 29 m [107, 6, 8, 9, 5, 7, 1, 23, 38, 34, 46, 20, 13, 5, 60, 51, 70, 42, 25, 33, 50, 80, 51, 11, 0, 47, 4, 47, 43]
 [ 2  4 14  4  6  2 10  8  6 10  6  2 10  6  2 12 10  2  4 12  8  4  6  6]
 5 29 m [107, 6, 8, 9, 5, 7, 1

## NOTE: Sorted by primorial coordinates
The constellations and their primorial prefixes are now sorted by their unique prefixes of primorial coordinates.
This data is saved to file so that we can start further explorations from here.

In [19]:
#  Commenting out this block to protect the existing files
'''
with open('Eng458_sorted.csv','w',newline='\n') as fp:
    writer = csv.writer(fp)
    writer.writerows(Eng_consts)

with open('Eng458_prefixes.csv','w',newline='\n') as fq:
    writerB = csv.writer(fq)
    writerB.writerows(Eng_pre_sorted)
'''

"\nwith open('Eng458_sorted.csv','w',newline='\n') as fp:\n    writer = csv.writer(fp)\n    writer.writerows(Eng_consts)\n\nwith open('Eng458_prefixes.csv','w',newline='\n') as fq:\n    writerB = csv.writer(fq)\n    writerB.writerows(Eng_pre_sorted)\n"

In [20]:
drivingterm59 =[2,4,14,4,6,2,10,2,6,6,10,6,2,10,6,2,12,10,2,4,12,2,6,4,6,6,6,8,6,6,4,6,8,6,4,14,16,6,6,8,10,2,4,8,6,6,6,
                10,2,10,6,2,10,8,4,6,14,6,4,2,6,22,2,4,2,12,10,8,4,6,2,16,2,4,6,8,6,4,12,2,10,2,10,6,8,6,10,6,2,6,4,2,
                6,10,8,4,2,10,8,6,10,2,4,6,8,10,12,2,10,6,2,10,2,12,4,2,4,8,10,6,6,6,8,4,8,18,4,2,6,4,8,10,6,6,6,2,6,10,
                6,14,4,2,4,2,24,6,4,8,10,2,4,12,14,16,8,4,6,2,4,8,16,14,16,2,10,8,4,6,2,12,10,2,4,2,4,6,8,4,2,10,8,6,6,
                6,4,6,8,4,6,2,18,16,6,2,6,6,10,6,8,4,2,4,12,2,10,2,4,12,2,10,2,4,6,8,6,6,6,4,6,18,2,4,8,10,6,12,2,6,6,
                4,6,14,10,2,10,14,4,6,6,2,6,6,16,2,6,12,10,8,4,2,6,18,4,6,2,10,18,2,4,8,6,4,2,6,12,10,6,6,6,2,10,2,10,
                6,8,16,6,14,4,2,4,14,4,20,6,6,6,12,6,6,4,2,10,12,2,16,8,10,6,6,2,10,2,6,4,18,2,4,6,14,4,6,2,12,12,4,2,
                12,6,4,2,4,12,14,4,2,12,6,10,6,6,12,2,6,4,6,8,4,8,4,14,12,10,2,6,6,4,2,4,6,2,12,6,12,10,6,2,4,8,6,10,6,
                8,6,6,10,6,14,4,6,8,6,10,2,18,10,6,6,2,10,18,2,12,4,2,10,14,4,14,10,2,10,18,2,4,2,4,8,6,4,6,6,6,8,12,10,
                2,6,4,6,2,22,2,10,6,2,6,6,16,8,4,2,10,6,8,4,2,6,12,6,4,6,6,6,8,6,4,2,10,2,12,4,2,10,2,12,10,14,4,2,4,6,8,
                6,4,12,6,8,4,2,4,2,12,6,4,12,6,2,6,10,2,4,6,8,4,2,4,2]
len(drivingterm59), sum(drivingterm59)

(478, 3242)

In [21]:
# Repeating the calculation of admissible instances
# in order to track the lengths of the inadmissible driving terms
# for each of the (458,3240)-counterexamples

i = 0
while (i < 58):
    target_s = Eng_consts[i]
    if target_s[0] == 2:
        drivingterm = np.array(drivingterm59[0:-1].copy())
    else:
        drivingterm = np.array(drivingterm59[1:].copy())
    ip = 17  # index for p=61 in smallp[]
    dt_len = len(drivingterm)
    while (dt_len > 458):
        pk = smallp[ip]
        rezlist = admissible(pk, target_s)
        rez = list(rezlist)[0]
        if (len(rezlist) > 1):
            print(f"Unexpected {i} {pk} length {dt_len}")
        j = 0
        while (j < (dt_len-1)):
            rez = (rez + drivingterm[j]) % pk
            if (rez == 0):     # fuse this and the following gap
                rez = drivingterm[j+1]
                drivingterm[j] = drivingterm[j]+drivingterm[j+1]
                drivingterm[j+1] = 0
                j += 1
            j += 1
        #
        drivingterm = drivingterm[ drivingterm > 0]
        dt_len = len(drivingterm)
        print(f" {i:2d} {ip:2d} {pk:3d} length {dt_len:3d}")
        ip += 1
        dt_len = len(drivingterm)
        
    i += 1

  0 17  61 length 472
  0 18  67 length 469
  0 19  71 length 467
  0 20  73 length 464
  0 21  79 length 464
  0 22  83 length 464
  0 23  89 length 462
  0 24  97 length 461
  0 25 101 length 460
  0 26 103 length 460
  0 27 107 length 460
  0 28 109 length 459
  0 29 113 length 458
  1 17  61 length 473
  1 18  67 length 470
  1 19  71 length 468
  1 20  73 length 465
  1 21  79 length 465
  1 22  83 length 465
  1 23  89 length 463
  1 24  97 length 461
  1 25 101 length 460
  1 26 103 length 460
  1 27 107 length 460
  1 28 109 length 459
  1 29 113 length 458
  2 17  61 length 473
  2 18  67 length 470
  2 19  71 length 468
  2 20  73 length 465
  2 21  79 length 465
  2 22  83 length 465
  2 23  89 length 463
  2 24  97 length 460
  2 25 101 length 460
  2 26 103 length 460
  2 27 107 length 460
  2 28 109 length 459
  2 29 113 length 458
  3 17  61 length 473
  3 18  67 length 470
  3 19  71 length 468
  3 20  73 length 465
  3 21  79 length 465
  3 22  83 length 465
  3 23  89

In [22]:
smallp[30:75]

array([127, 131, 137, 139, 149, 151, 157, 163, 167, 173, 179, 181, 191,
       193, 197, 199, 211, 223, 227, 229, 233, 239, 241, 251, 257, 263,
       269, 271, 277, 281, 283, 293, 307, 311, 313, 317, 331, 337, 347,
       349, 353, 359, 367, 373, 379])

In [23]:
smallp[0:18],smallp[29:40],smallp[450:463]

(array([ 2,  3,  5,  7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59,
        61]),
 array([113, 127, 131, 137, 139, 149, 151, 157, 163, 167, 173]),
 array([3187, 3191, 3203, 3209, 3217, 3221, 3229, 3251, 3253, 3257, 3259,
        3271, 3299]))

## $\Delta \Phi$ to picture $p$-rough numbers

The count of $p$-rough numbers up through $x$ is denoted $\Phi(x,p)$.
The graph of $\Phi(x,p)$ has a line of symmetry
$$ \tilde{\Phi} = \frac{\phi(p^\#)}{p^\#} x = \frac{1}{\mu} x $$
where $\mu = \frac{p^\#}{\phi(p^\#)}$ is the mean size of the gaps in
$\mathcal{G}(p^\#)$.

So we work with $\Delta \Phi(x,p)$, which measures the deviation of $\Phi(x,p)$ around its line of symmetry.
$$ \Delta \Phi(x,p) = \Phi(x,p) - \frac{1}{\mu} x$$

The deviations of the $p$-rough numbers from their line of symmetry
$$\tilde{\Phi} = \frac{1}{\mu}x$$ 
are periodic and bounded.  Thus the behavior of $\Phi(x,p)$ is completely described by the behavior of $\Delta \Phi(x,p)$
over the first cycle $\mathcal{G}(p^\#)$, or using the rotational symmetry around $x=1+\frac{p^\#}{2}$ over the first
half of this cycle.

$\Delta \Phi(x,p)$ provides a good visualization of the $p$-rough numbers.

To use this visualization here, we use the average prime gap over the first $462$ primes, $p_{462}=3271$

| $k$ | $456$ | $457$ | $458$ | $459$ | $460$ | $461$ | $462$ | $463$ |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| $p_k$ | $3221$ | $3229$ | $3251$ | $3253$ | $3257$ | $3259$ | $3271$ | $3299$ |
| $\max|s|$ |  | $3236$ | $3240$ | $3242$ | $3276$ | | | |



In [24]:
# average gap size for 
mu_gap = 3271/462
mu_recip = -1/mu_gap
print("mu",mu_gap, "neg reciprocal (slope)", mu_recip)

mu 7.08008658008658 neg reciprocal (slope) -0.14124121063894834


In [25]:
# create the constellation of prime gaps
prime_constellation = smallp[1:462]-smallp[0:461]
prime_constellation = np.concatenate(([2], prime_constellation))


In [26]:
# this function interleaves arr1 with arr2, which we need for plotting the vertical segments
def interleave_np(arr1, arr2):
    stacked_arr = np.stack((arr1, arr2),axis=1)
    return stacked_arr.flatten().tolist()

In [28]:
# Plotting pi function vs Engelsma constellation (459,3242) 
# Interleaving data to get the stepped graph, rendering the vertical segments
# 
def Eng458plot(input_dex,printflag):
    input_s = Eng_consts[input_dex][0:]

    # data for plotting the prime constellation
    xp = smallp[0:462] # 462 values from 2 to 3271
    xp = interleave_np(xp,xp)  # ... doubled up...
    xp = np.concatenate(([0],xp)) # 1+2*462 values from 0 to 3271

    delpi = np.zeros(462) # we will calculate the lower points first
    i=1
    delpi[0] = 2*mu_recip
    while (i < 462):
        delpi[i] = delpi[i-1] + 1 + prime_constellation[i]*mu_recip
        i += 1

    delpi = interleave_np(delpi, (delpi + 1)) # interleave the lower points and upper points for the vertical segments
    delpi = np.concatenate(([0],delpi))

    pidf = pd.DataFrame({'x':xp, 'picnt': delpi})

    # data for plotting the input constellation
    xcons = np.cumsum(input_s)
    xcons = interleave_np(xcons, xcons)
    xcons = np.concatenate(([0],xcons))

    delconst = np.zeros(len(input_s))
    delconst[0] = input_s[0]* mu_recip
    i=1
    while (i < len(input_s)):
        delconst[i] = delconst[i-1] + 1 + input_s[i]*mu_recip
        i += 1

    delconst = interleave_np(delconst, (delconst+1))
    delconst = np.concatenate(([0],delconst))

    data_sample = {'x': xcons, 'DelPhi': delconst}
    df = pd.DataFrame(data_sample)

    # plotting the two curves
    plt.clf()
    fig, ax = plt.subplots()
    fig.set_size_inches(14,9)
    ax.set_title(f"Engelsma(458, 3240) #{input_dex} vs $\pi(n)$ as segments of $\Delta \Phi(x,\mu)$")
    ax.grid(axis='y', color='#080408', lw=0.125 )
     
    ax.plot(xcons, delconst, color='#2222AF', lw=0.125, label='DelPhi')
    ax.plot(xp, delpi, color='#AF0000', lw=0.25, label='pi(n)')
# ax.set_ylim(-8,12)
    if printflag:
        pltsavename = 'Eng458_'+ str(input_dex)+'_vpi.png'
        plt.savefig(pltsavename, dpi=450)
        
    plt.show()

xEng458Select = widgets.IntSlider(value=29, min=0, max=115, description="Which (458,3240)", 
                                  layout=widgets.Layout(width='80%'), style={'description_width':'90pt'}, disabled=False)

xEng458Print = widgets.ToggleButton(value=False, description='Print Figure', disabled=False, button_style='', tooltip='Description', icon='check')
interact(Eng458plot, input_dex=xEng458Select, printflag=xEng458Print)

interactive(children=(IntSlider(value=29, description='Which (458,3240)', layout=Layout(width='80%'), max=115,…

<function __main__.Eng458plot(input_dex, printflag)>

## Explore the constellations one at a time
We look at each of the 116 $(458,3240)$-counterexamples.  We have the constellation, its numbers of admissible residues across
several primes, and the corresponding unique primorial coordinates starting in ${\mathcal G}(11^\#)$.

For each constellation we find all of the extensions to its primorial coordinates through the next several primes.  This exhaustive search
is limited by the sheer number of possibilities.  We then take an opportunistic random approach to searching the larger space of
extensions.  We are looking for long sequences of coordinates $m_k=0$.

In [29]:
len(Eng_pre_sorted[25]), Eng_pre_sorted[25][0:12]

(28, [107, 6, 8, 9, 5, 7, 1, 23, 38, 34, 46, 20])

In [30]:
Eng458_prefixesB = []
with open('Eng458_prefixes.csv', 'r') as fqtr:
    pref_reader = csv.reader(fqtr)
    for row in pref_reader:
        Eng458_prefixesB.append([int(x) for x in row])
fqtr.close()

In [31]:
len(Eng458_prefixesB[25])

28

In [32]:
Eng_consts[25][0:]

array([ 2,  4, 14,  4,  6,  2, 10,  2,  6,  6, 10,  6,  2, 10,  6, 14, 10,
        2,  4, 12,  2,  6,  4,  6,  6,  6,  8,  6,  6,  4,  6,  8,  6, 18,
       16,  6,  6,  8, 10,  2,  4,  8,  6, 12, 10,  2, 10,  6,  2, 10,  8,
        4, 20,  6,  4,  2,  6, 22,  2,  4,  2, 12, 10,  8,  4,  8, 16,  2,
        4,  6,  8,  6,  4, 12,  2, 10,  2, 10,  6,  8,  6, 10,  6,  2,  6,
        4,  8, 10,  8,  4,  2, 10,  8,  6, 10,  2,  4,  6,  8, 10, 14, 10,
        6, 12,  2, 12,  4,  2,  4,  8, 10,  6,  6,  6,  8,  4,  8, 18,  4,
        2,  6,  4,  8, 16,  6,  6,  2,  6, 10,  6, 18,  2,  4,  2, 24,  6,
       12, 10,  2,  4, 12, 14, 16,  8,  4,  6,  2,  4,  8, 16, 14, 16,  2,
       10,  8,  4,  6,  2, 12, 10,  2,  4,  2,  4,  6, 12,  2, 10,  8,  6,
        6, 10,  6,  8,  4,  6,  2, 18, 16,  6,  2,  6,  6, 10,  6,  8,  4,
        2,  4, 12,  2, 10,  2,  4, 12,  2, 10,  2,  4,  6,  8,  6,  6,  6,
        4,  6, 18,  2,  4,  8, 10, 18,  2,  6,  6,  4,  6, 14, 10,  2, 10,
       14,  4,  6,  6,  2

In [33]:
Eng458_sortedB = []
with open('Eng458_sorted.csv', 'r') as fqtr:
    ssort_reader = csv.reader(fqtr)
    for row in ssort_reader:
        Eng458_sortedB.append([int(x) for x in row])
fqtr.close()

In [34]:
s25 = np.array(Eng_consts[25][0:])
gaps, gapcounts = np.unique(s25, return_counts= True)
gaps, gapcounts

(array([ 2,  4,  6,  8, 10, 12, 14, 16, 18, 20, 22, 24]),
 array([ 87,  75, 119,  40,  57,  36,  15,  11,  12,   3,   2,   1]))

In [35]:
# Computing the second factor for Q for s4
s25 = np.array(Eng_consts[25])
iq = 1
winfB = 1
while (smallp[iq] < 458):
    iq += 1
while (smallp[iq] <= 1621):
    pq = smallp[iq]
    rezq = admissible(pq,s25)
    lenups = len(rezq)
    if (lenups > (pq-459)):
        print(f"{iq:3d} {pq:4d} factor {lenups:4d}/{pq-459}")
        winfB = winfB * (lenups/(pq-459))
    iq += 1
print(f"winfty factor: {winfB}")

 88  461 factor  115/2
 89  463 factor  128/4
 90  467 factor  127/8
 91  479 factor  135/20
 92  487 factor  131/28
 93  491 factor  148/32
 94  499 factor  141/40
 95  503 factor  149/44
 96  509 factor  148/50
 97  521 factor  159/62
 98  523 factor  151/64
 99  541 factor  167/82
100  547 factor  180/88
101  557 factor  181/98
102  563 factor  187/104
103  569 factor  197/110
104  571 factor  196/112
105  577 factor  204/118
106  587 factor  202/128
107  593 factor  218/134
108  599 factor  220/140
109  601 factor  216/142
110  607 factor  224/148
111  613 factor  228/154
112  617 factor  229/158
113  619 factor  240/160
114  631 factor  248/172
115  641 factor  251/182
116  643 factor  258/184
117  647 factor  256/188
118  653 factor  264/194
119  659 factor  266/200
120  661 factor  270/202
121  673 factor  282/214
122  677 factor  277/218
123  683 factor  284/224
124  691 factor  297/232
125  701 factor  299/242
126  709 factor  307/250
127  719 factor  319/260
128  727 factor  

In [36]:
smallp[254]

np.int64(1613)

In [37]:
np.log10(len(Eng458_sortedB[25]))

np.float64(2.660865478003869)

In [38]:
# We start with the unique prefixes in Eng_pre_sorted and the constellations in Eng_const
# The array smallp starts at p=2, and p0 for the prefixes is p=11.  The primorial coordinate m[k] corresponds to smallp[k+4]

debug_verbose = True
saveto_file = True     # flag for writing the _results and _mzeros files
ic = int(0)  # index for the constellation

while (ic < len(Eng458_sortedB)):
    pref_len = len(Eng458_prefixesB[ic])  # for the ic-th counterexample - 
    Eng458s = Eng458_sortedB[ic]             # get the constellation
    mprefix = Eng458_prefixesB[ic]        # get the unique prefix for the primorial coordinates

    # for this constellation and prefix, determine the admissible residues for an exhaustive search 
    #  of up to 20M examples each
    rezlist = []
    ip0 = 4 + pref_len
    while (smallp[ip0] < 5000): 
        p = smallp[ip0]
        rezp = admissible(p,Eng458s)
        rezlist.append(rezp)
        ip0 += 1

    # Summarize num_admissible across the primes p
    num_admissible = [len(rezlist[i]) for i in range(len(rezlist))]
    num_admissible = np.array(num_admissible)

    if debug_verbose:
        print(f"{ic:2d} gamma0 {Eng458s[0]:4d} len_prefix {pref_len:2d} num_adm {num_admissible[0:60]}")
        # for element in rezlist:
            # print(f" {list(element)[0]}", end=' ')
        # print()

    # We extend the primorial coordinates 
    num_instances = 1
    m_ext_list = []
    k = 0      # index for pk beyond the prefix, e.g. for num_admissible and rezlist

    
    if debug_verbose:
        print(f"k {k} num_admissible {num_admissible[k]} total instances {num_instances}", end='\r')

    # initiate the extensions in first iteration on k
    ik = k+pref_len   # index for smallp[] corresponding to k
    pk = smallp[ik+4]
    num_instances = num_admissible[k]
    i = 0
    while (i < num_admissible[k]):
        m_ext_list.append(list(mprefix))

        target_residue = list(rezlist[k])[i]
        # calculate residue r0 modulo pk
        # set initial conditions for this loop
        j = 1
        r0 = m_ext_list[i][0] % pk
        rpml = 2310 % pk  # initialize this factor at 11#
        while (j < ik):  # calculate the residue for the primorial expansion
            r0 = (r0 + m_ext_list[i][j] * rpml) % pk
            rpml = (rpml * smallp[j+4]) % pk
            j += 1
        # calculate next primorial coefficient mk
        mk = primorialm(ik,target_residue,r0)
        (m_ext_list[i]).append(int(mk))

        if debug_verbose:
            print(f"k {k} {ik} p {smallp[ik+4]} i {i} targetr {target_residue} r0 {r0}")
            for element in m_ext_list[i]:
                print(f"{element:3d}", end=" ")
            print()

        i += 1

    # now iterate k through the primes pk.  
    #  k is the index in num_admissible[] which starts after the unique prefix
    #  ik is the corresponding index in smallp[], as an offset from 11
    k += 1
    ik += 1

    while (num_instances < 4000000):
        num_instances *= num_admissible[k]
        pk = smallp[ik+4]
    
        iprefix=0
        num_prefixes = len(m_ext_list)
        # iprefix is the index through existing primorial expansions
        # i is the index through admissible residues at this stage
        
        while (iprefix < num_prefixes):
            # calculate r0 mod pk
            j = 1
            r0 = m_ext_list[iprefix][0] % pk
            rpml = 2310 % pk  # initialize this factor at 11#
            while (j < ik):  # the expansion for r 
                r0 = (r0 + m_ext_list[iprefix][j] * rpml) % pk
                rpml = (rpml * smallp[j+4]) % pk
                j += 1

            # extend the first copy in place
            # save the prefix
            m_start = list(m_ext_list[iprefix].copy())
            # calculate next primorial coefficient mk
            target_residue = list(rezlist[k])[0]
            mk = primorialm(ik,target_residue,r0)
            (m_ext_list[iprefix]).append(int(mk))

            if ((iprefix % 4096)==0):
                print(f"ic {ic} k {k} {ik} p {pk} i {iprefix} targetr {target_residue} r0 {r0} length {len(m_ext_list[iprefix])}", end='\r')
        
            i=1
            while (i < num_admissible[k]):   # for this prefix append copies with all the admissible coordinates for this prime
                icopy = len(m_ext_list)
                m_ext_list.append(m_start.copy())  # icopy is the index for this copy
                target_residue = list(rezlist[k])[i]
                mk = primorialm(ik,target_residue,r0)
                (m_ext_list[icopy]).append(int(mk))
                # print(f"Copy {icopy} k {k} p {smallp[k+4]} i {iprefix} {i} targetr {target_residue} r0 {r0} length {len(gammam_list[icopy])}")

                i += 1

            iprefix += 1  # next prefix

        k += 1     # next prime - depth of search
        ik += 1

    # Process and record the results for this constellation
    filename = 'Eng458_' + str(ic) + '_results.csv'
    k -= 1
    ik -= 1
    
    # record ic, prefix_len, p0, k, pk, num_instances, num_admissible
    # XXXQHERE [6/22] - how much of num_admissible to save?
    if saveto_file:
        results_list = [ic, pref_len, smallp[4+pref_len], k, smallp[k+pref_len+4], num_instances, num_admissible]
    
        # Factors of asymptotic relative population 
        # - up through J+1=459, and up through |s|/2 = 1620
        # We calculate the log of this factor, anticipating floating point overflow
        ws_J_log = 0
        i = 0    # indexing here: admissible 0 == primes 4+pref_len
        pk = smallp[i + 4 + pref_len]
        while (pk <= 459):
            ws_J_log += np.log10(num_admissible[i])
            i += 1
            pk = smallp[i + 4 + pref_len]

        ws_s_log = 0
        while (pk <= 1620):
            ws_s_log += np.log10(num_admissible[i] / (pk-459))
            i += 1
            pk = smallp[i + 4 + pref_len]

        results_list.append([ws_J_log, ws_s_log])

        with open(filename, 'w', newline='\n') as fptr:
            writer = csv.writer(fptr)
            writer.writerow(results_list)
        fptr.close()
    
    # Identify instances whose primorial coordinates end in sequences of 0's
    num_mext = len(m_ext_list)
    zero_list = []
    imext = 0
    while (imext < num_mext):
        j = len(m_ext_list[imext])-1
        while ( m_ext_list[imext][j] == 0):
            j -= 1
        nzer = len(m_ext_list[imext]) - 1 - j
        if (nzer > 0):
            zero_list.append([nzer,imext])
        imext += 1

    if saveto_file:
        if (len(zero_list)>0):
            sorted_zero_list = sorted(zero_list, reverse=True)
            max_zeros = sorted_zero_list[0][0]
            print(f"ic {ic} max0s {max_zeros} Saving {len(zero_list)} extensions out of {num_mext}")
            m_zeros = [] 
            j=0
            while (j < len(sorted_zero_list)):
                imext = sorted_zero_list[j][1]
                entry = [ sorted_zero_list[j], m_ext_list[imext]]
                m_zeros.append(entry)
                j += 1

            filename = 'Eng458B_' + str(ic) + '_mzeros.csv'

            with open(filename, 'w', newline='\n') as fptr:
                writer = csv.writer(fptr)
                writer.writerows(m_zeros)
            fptr.close()


    ic +=1   # Loop into next constellation (458,3240)



 0 gamma0    2 len_prefix 29 num_adm [  2   1   1   3   1   2   3   4   3   9   5   5   5  10   8  13   7  10
   9  18  14  17  25  21  17  23  28  22  28  31  37  34  39  38  50  53
  49  55  59  64  72  69  73  81  89  85  87  95  96 104 103 103 108 112
 112 116 132 126 134 130]
k 0 29 p 139 i 0 targetr 24 r0 91ces 1
107   6   8   9   5   7   1  23  38  34  46  20  13   4   4  53  64  11  39  27  17  44   1  78  22 108  29 112  72 132 
k 0 29 p 139 i 1 targetr 90 r0 91
107   6   8   9   5   7   1  23  38  34  46  20  13   4   4  53  64  11  39  27  17  44   1  78  22 108  29 112  72 114 
ic 0 max0s 2 Saving 22910 extensions out of 4860000 4343
 1 gamma0    2 len_prefix 29 num_adm [  2   1   1   3   2   3   2   4   3   9   5   5   6  11   8  13   8  10
  10  17  16  16  24  19  18  22  28  22  29  32  38  35  39  39  48  50
  50  57  60  66  69  66  73  83  88  81  89  95  97 105  99 103 109 113
 110 116 129 123 133 131]
k 0 29 p 139 i 0 targetr 24 r0 136es 1
107   6   8   9   5   7  

In [39]:
smallp[0:50]

array([  2,   3,   5,   7,  11,  13,  17,  19,  23,  29,  31,  37,  41,
        43,  47,  53,  59,  61,  67,  71,  73,  79,  83,  89,  97, 101,
       103, 107, 109, 113, 127, 131, 137, 139, 149, 151, 157, 163, 167,
       173, 179, 181, 191, 193, 197, 199, 211, 223, 227, 229])

## A couple of reference indices for smallp[]:

smallp[33] = 139

smallp[87] = 457  (then 461)

smallp[255] = 1619 (then 1621)

In [40]:
smallp[75:256]

array([ 383,  389,  397,  401,  409,  419,  421,  431,  433,  439,  443,
        449,  457,  461,  463,  467,  479,  487,  491,  499,  503,  509,
        521,  523,  541,  547,  557,  563,  569,  571,  577,  587,  593,
        599,  601,  607,  613,  617,  619,  631,  641,  643,  647,  653,
        659,  661,  673,  677,  683,  691,  701,  709,  719,  727,  733,
        739,  743,  751,  757,  761,  769,  773,  787,  797,  809,  811,
        821,  823,  827,  829,  839,  853,  857,  859,  863,  877,  881,
        883,  887,  907,  911,  919,  929,  937,  941,  947,  953,  967,
        971,  977,  983,  991,  997, 1009, 1013, 1019, 1021, 1031, 1033,
       1039, 1049, 1051, 1061, 1063, 1069, 1087, 1091, 1093, 1097, 1103,
       1109, 1117, 1123, 1129, 1151, 1153, 1163, 1171, 1181, 1187, 1193,
       1201, 1213, 1217, 1223, 1229, 1231, 1237, 1249, 1259, 1277, 1279,
       1283, 1289, 1291, 1297, 1301, 1303, 1307, 1319, 1321, 1327, 1361,
       1367, 1373, 1381, 1399, 1409, 1423, 1427, 14

In [41]:
i=0
while (smallp[i] < 1620):
    i += 1
print(f"{i-1} p {smallp[(i-1):(i+1)]}")

255 p [1619 1621]


## Data files _results and _mzeros
The search through primorial coordinates above produces two sets of output files:  <i>Eng458_xx_results.csv</i> and <i>Eng458B_xx_mzeros.csv</i>.  At this point, the search is breadth-first and exhaustive.  So not very deep.  

In order to pursue the survivial of any incidence of these (458,3240)-counterexamples, we have to switch to a greedy depth-first
approach.  Survival is indicated by a <i>long</i> sequence of consecutive primorial coordinates $m_k = 0$.  So we start by considering
those constellations with primorial expansions from the current search that end with at least one $m_k=0$.

<i>Eng458B_xx_results.csv</i> contains summary notes about the search over the (458,3240)-counterexample of index <i>xx</i> in the file 
<i>Eng458_sorted.csv</i>  The first few fields in the <i>_results</i> file are the index <i>xx</i> of the counterexample; the length of
the unique prefix of the primorial coordinates, starting at $p_{0}=11$; the prime $p_{k_0}$ just beyond this unique prefix, where there is more than one admissible residue; the depth $k$ of the breadth-first search into extensions of the prefix; the prime corresponding to 
the last $m_k$ recorded; and the number of admissible instances searched.  Then the array of the numbers of admissible instances $\bmod p$
is listed, starting at $p_{k_0}$.  After this array we list the two factors (in log-base-10) of the asymptotic relative population
for this constellation:
$$ w_{s,J}(\infty) \; = \; \prod_{p \le J+1} (p - \nu(p)) \cdot \prod_{p > J+1} \frac{p - \nu(p)}{p - J-1}$$

<i>Eng458_xx_mzeros.csv</i> contains data about the extensions of the primorial coordinates for the (458,3240)-counterexample of
index <i>xx</i> in the file <i>Eng458_sorted.csv</i>  Each row of data starts with a two-element array:  
the number of terminal zeroes for this extension, and the index of the extension in the breadth-first search.  
This is followed by the primorial coordinates from $p_0=11$.

## Reading _results.csv files
If we use csv.reader and read in the <i>_results.csv</i> file row-by-row, say to <i>str_data</i> then we have

str_data[0][0] = ic, index of the counterexample

str_data[0][1] = lenp, the length of the unique prefix of primorial coordinates starting at $p_0=11$, which has index 4 in <i>smallp[]</i>

str_data[0][2] = $p_{k_0}$, the prime just beyond the unique prefix

str_data[0][3] = depth, the depth of the breadth-first search beyond the unique prefix

str_data[0][4] = $p_k$, the prime corresponding to the last $m_k$ recorded

str_data[0][5] = numsamples, the number of instances searched

str_data[0][6] = numadmissible[], a list of the numbers of admissible instances starting at $p_{k_0}$

str_data[0][7] = log_winfJ, log_winfQ, the factors for the asymptotic relative population for this constellation, stored as log10 of the floating-point values.

Using csv.reader these fields are read in as strings and must be converted back to numeric form for further processing.

## Counterexample surviving the sieve
Every Engelsma counterexample (458,3240) has a unique image up through ${\mathcal G}(131^\#)$.  
The constellation itself first appears in 
${\mathcal G}(113^\#)$, and there are no longer driving terms.

For $s_{31}$ and $s_{84}$ for example, by ${\mathcal G}(457^\#)$ there are $5.18168 \; E72$ images of this constellation.
Over $461 \le p_k \le 1619$ the relative population is 
$$\prod_{461}^{1619} \frac{q-\nu(q)}{q-459} \; = \; 5.01281 \;E16$$

Thus the asymptotic relative population of these Engelsma (458,3240)-counterexamples is
$$w_{s,458}(\infty) = 2.597479 \cdot E89$$

In [42]:
# Summarizing the asymptotic relative populations for the (458,3240)-counterexamples
Eng458_winf = np.zeros((116,4),dtype=float)

ic = 0
while (ic < 116):
    # extract the data from the results file for this constellation
    filename = 'Eng458_' + str(ic) + '_results.csv'
    Eng458_str_data = []
    with open(filename, 'r') as fqtr:
        data_reader = csv.reader(fqtr)
        for row in data_reader:
            Eng458_str_data.append([x for x in row])
    fqtr.close()

    # parse the values for log10(winfty) where winfty is broken into its two factors
    #  winfty(q <= J+1) and wfinty(q | Q and q> J+1)
    str_winf = Eng458_str_data[0][7]
    i0=0
    while (str_winf[i0] != '('):
        i0 += 1
    i1 = i0
    while (str_winf[i1] != ')'):
        i1 += 1
    logwsinf_J = float(str_winf[(i0+1):i1])
    i0 = i1
    while (str_winf[i0] != '('):
        i0 += 1
    i1 = i0
    while (str_winf[i1] != ')'):
        i1 += 1
    logwsinf_Q = float(str_winf[(i0+1):i1])

    winfJ = 10**logwsinf_J
    winfQ = 10**logwsinf_Q
    winf = winfJ * winfQ

    print(f"{ic:3d}  {len(Eng458_str_data[0])} winfJ {winfJ:10.5e} winfQ {winfQ:10.5e} winf {winf:10.5e}")

    Eng458_winf[ic,0] = ic
    Eng458_winf[ic,1] = winfJ
    Eng458_winf[ic,2] = winfQ
    Eng458_winf[ic,3] = winf
    
    ic += 1


  0  8 winfJ 1.98400e+72 winfQ 5.54722e+16 winf 1.10057e+89
  1  8 winfJ 6.14865e+72 winfQ 5.00098e+16 winf 3.07492e+89
  2  8 winfJ 6.47823e+72 winfQ 4.90217e+16 winf 3.17574e+89
  3  8 winfJ 3.96920e+72 winfQ 5.02201e+16 winf 1.99333e+89
  4  8 winfJ 2.93968e+72 winfQ 4.80101e+16 winf 1.41134e+89
  5  8 winfJ 2.27416e+72 winfQ 5.03179e+16 winf 1.14431e+89
  6  8 winfJ 9.54496e+72 winfQ 5.70357e+16 winf 5.44404e+89
  7  8 winfJ 5.18838e+72 winfQ 5.73213e+16 winf 2.97405e+89
  8  8 winfJ 1.13295e+73 winfQ 5.61135e+16 winf 6.35738e+89
  9  8 winfJ 5.93734e+72 winfQ 5.65592e+16 winf 3.35811e+89
 10  8 winfJ 6.79928e+72 winfQ 5.70411e+16 winf 3.87839e+89
 11  8 winfJ 1.35673e+73 winfQ 5.58332e+16 winf 7.57505e+89
 12  8 winfJ 1.44980e+73 winfQ 5.62971e+16 winf 8.16195e+89
 13  8 winfJ 1.50261e+73 winfQ 5.64817e+16 winf 8.48702e+89
 14  8 winfJ 1.72225e+73 winfQ 5.72471e+16 winf 9.85938e+89
 15  8 winfJ 1.91710e+73 winfQ 5.53179e+16 winf 1.06050e+90
 16  8 winfJ 1.05365e+73 winfQ 5.62815e+

In [43]:
Eng458_winf[0:10,:]

array([[0.00000000e+00, 1.98400002e+72, 5.54722248e+16, 1.10056895e+89],
       [1.00000000e+00, 6.14864798e+72, 5.00097647e+16, 3.07492438e+89],
       [2.00000000e+00, 6.47822921e+72, 4.90217174e+16, 3.17573921e+89],
       [3.00000000e+00, 3.96919580e+72, 5.02200575e+16, 1.99333241e+89],
       [4.00000000e+00, 2.93967815e+72, 4.80101052e+16, 1.41134257e+89],
       [5.00000000e+00, 2.27416461e+72, 5.03179286e+16, 1.14431252e+89],
       [6.00000000e+00, 9.54495907e+72, 5.70357484e+16, 5.44403884e+89],
       [7.00000000e+00, 5.18838365e+72, 5.73213063e+16, 2.97404928e+89],
       [8.00000000e+00, 1.13295038e+73, 5.61135142e+16, 6.35738270e+89],
       [9.00000000e+00, 5.93734123e+72, 5.65591505e+16, 3.35810976e+89]])

In [44]:
# save the asymptotic relative populations, including their J-factors and Q-factors to file
filename = 'Eng458_winf.csv'

np.savetxt(filename, Eng458_winf, delimiter=',')
    

In [45]:
# from the _mzeros files, find the smallest initial generator for each constellation

Eng458_gamma0 = []

ic = 0
while ic < 116:  # ic is the index for the (458,3240)-counterexample
    filename = 'Eng458B_' + str(ic) + '_mzeros.csv'

    max_zeros = -1
    Eng458_m_data = []
    with open(filename, 'r') as fqtr:
        data_reader = csv.reader(fqtr)
        for row in data_reader:
            Eng458_m_data.append([x for x in row])
    fqtr.close

    # We move through
    num_prefixes = len(Eng458_m_data)
    ip=1
    # initialize mingamma0 as the first prefix
    mingamma0 = np.fromstring(Eng458_m_data[0][1][1:-1], dtype=int, sep=',')
    param_array = np.fromstring(Eng458_m_data[0][0][1:-1], dtype=int, sep=',')
    max_zeros = param_array[0]

    while ip < num_prefixes:
        # get the number of terminal zeros
        param_array = np.fromstring(Eng458_m_data[ip][0][1:-1], dtype=int, sep=',')
        nzeros = param_array[0]
        
        if nzeros > max_zeros:
            max_zeros = nzeros  # This shouldn't happen...
            print(f"Surprise: const {ic} prefix {ip} nzeros {nzeros}")
            
        if nzeros == max_zeros:
            # get the data for the primorial expansion
            primorial_coords = np.fromstring(Eng458_m_data[ip][1][1:-1], dtype=int, sep=',')
            imk = len(primorial_coords)-1
            while primorial_coords[imk] == mingamma0[imk]:
                imk -= 1
            # imk marks the first coordinate from the back where the two prefixes disagree
            if primorial_coords[imk] < mingamma0[imk]:  # a new smallest instance
                mingamma0 = primorial_coords.copy()
                # print(f"{ic:3d} {ip:3d} nzeros {nzeros} last coeff {minmk:3d}")

        if nzeros < max_zeros:
            break

        ip += 1

    Eng458_gamma0.append(mingamma0.copy())
    print(f"{ic:3d} len {len(mingamma0):4d} nzeros {max_zeros} last coeffs {mingamma0[-10:]}")

    ic += 1   # next (458,3240)-counterexample

  0 len   43 nzeros 2 last coeffs [ 51 166  40 170  79 189 130   2   0   0]
  1 len   43 nzeros 3 last coeffs [50 62 83 97 54 60  6  0  0  0]
  2 len   43 nzeros 3 last coeffs [153 128  12 107 146 137   7   0   0   0]
  3 len   43 nzeros 3 last coeffs [103  26  53 148  94   7  87   0   0   0]
  4 len   43 nzeros 2 last coeffs [ 51  73  60  31 125 138 132   6   0   0]
  5 len   43 nzeros 2 last coeffs [155  52 110 127  68 152   9   5   0   0]
  6 len   43 nzeros 3 last coeffs [ 87 161  30  28 172 158   6   0   0   0]
  7 len   43 nzeros 3 last coeffs [139  72  39  65  81 139  68   0   0   0]
  8 len   43 nzeros 2 last coeffs [ 58  58   1  85 164 104 105   1   0   0]
  9 len   43 nzeros 3 last coeffs [141  35 157  16 177  59  62   0   0   0]
 10 len   43 nzeros 2 last coeffs [ 45 145   7  15 147 125 154   2   0   0]
 11 len   43 nzeros 3 last coeffs [137   4  63  18 109  52  77   0   0   0]
 12 len   43 nzeros 3 last coeffs [ 68  94  61 151  49  39 126   0   0   0]
 13 len   43 nzeros 3 

In [46]:
Eng458_gamma0[0]

array([107,   6,   8,   9,   5,   7,   1,  23,  38,  34,  46,  20,  13,
         4,   4,  53,  64,  11,  39,  27,  17,  44,   1,  78,  22, 108,
        29, 112,  72, 114,  82,  41,  69,  51, 166,  40, 170,  79, 189,
       130,   2,   0,   0])

In [47]:
# calculate the minimum initial generator for each constellation
ic = 0 

min_gamma0 = np.zeros(116)

while ic < len(Eng458_gamma0):
    pprimorial = 2310.0
    gamma0 = float(Eng458_gamma0[ic][0])
    ip = 5  # smallp[4] = 11
    imk = 1
    while (imk < len(Eng458_gamma0[ic])):
        gamma0 += float(Eng458_gamma0[ic][imk])*pprimorial
        pprimorial *= smallp[ip]
        ip += 1
        imk += 1

    min_gamma0[ic] = gamma0
    ic += 1


In [48]:
min_gamma0

array([5.32963344e+77, 6.51082613e+75, 7.96005993e+75, 8.97283125e+76,
       1.33060072e+78, 1.00491237e+78, 7.04327323e+75, 7.08533910e+76,
       3.07772385e+77, 6.42390991e+76, 5.57361351e+77, 7.96626873e+76,
       1.30104519e+77, 1.17597124e+77, 3.12381431e+76, 3.91238265e+76,
       1.00401970e+77, 6.31170678e+76, 1.73055459e+77, 8.48292648e+75,
       6.25014634e+76, 4.61731775e+76, 4.33457778e+74, 9.73312725e+73,
       2.49356009e+77, 6.68416502e+76, 3.34161041e+76, 8.00862288e+75,
       1.01283119e+77, 5.32963344e+77, 6.51082613e+75, 7.96005993e+75,
       8.97283125e+76, 1.33060072e+78, 1.00491237e+78, 7.04327323e+75,
       7.08533910e+76, 3.07772385e+77, 6.42390991e+76, 5.57361351e+77,
       7.96626873e+76, 1.30104519e+77, 1.17597124e+77, 3.12381431e+76,
       3.91238265e+76, 1.00401970e+77, 6.31170678e+76, 1.73055459e+77,
       8.48292648e+75, 6.25014634e+76, 4.61731775e+76, 4.33457778e+74,
       9.73312725e+73, 2.49356009e+77, 6.68416502e+76, 3.34161041e+76,
      

In [64]:
Eng458_gamma0[0]

array([107,   6,   8,   9,   5,   7,   1,  23,  38,  34,  46,  20,  13,
         4,   4,  53,  64,  11,  39,  27,  17,  44,   1,  78,  22, 108,
        29, 112,  72, 114,  82,  41,  69,  51, 166,  40, 170,  79, 189,
       130,   2,   0,   0])

In [66]:
# export these results
with open('Eng458_minprefixes.csv', 'w', newline='\n') as fptr:
    writer = csv.writer(fptr)
    writer.writerows(Eng458_gamma0)
fptr.close()
np.savetxt('Eng458_mingamma0.csv', min_gamma0, delimiter=',')

## Notes on search: J=458 vs J=459
For our breadth-first search, we do not find any instance of a (458,3240)-counterexample that is not part of a (459,3242)-counterexample.
Here we check the numbers of admissible instances to see where we might first find a (458,3240)-counterexample by itself.

In [67]:
ic458 = 23

# The indices occur in blocks
if ic458 < 29:
    ic459 = ic458
elif ic458 < 58:
    ic459 = ic458-29
elif ic458 < 87:
    ic459 = ic458-29
elif ic458 < 116:
    ic459 = ic458-58

filename = 'Eng458_' + str(ic458) + '_results.csv'
Eng458_str_data = []
with open(filename, 'r') as fqtr:
    data_reader = csv.reader(fqtr)
    for row in data_reader:
        Eng458_str_data.append([x for x in row])
fqtr.close()

filename = 'Eng459_' + str(ic459) + '_results.csv'
Eng459_str_data = []
with open(filename, 'r') as fqtr:
    data_reader = csv.reader(fqtr)
    for row in data_reader:
        Eng459_str_data.append([x for x in row])
fqtr.close()



In [74]:
Eng458_str_data[0][0], Eng458_str_data[0][1], Eng458_str_data[0][2], Eng458_str_data[0][3], Eng458_str_data[0][4], Eng458_str_data[0][5], Eng458_str_data[0][7]

('23',
 '28',
 '137',
 '13',
 '199',
 '4032000',
 '[np.float64(73.58791938191358), np.float64(16.73668688519735)]')

In [75]:
Eng458_str_data[0][6]

'[   2    2    1    1    3    2    2    2    5    4   10    6    5    7\n   12   10   13    7   12    9   18   16   18   26   20   17   22   29\n   23   28   31   39   35   38   40   48   53   50   55   59   63   69\n   65   74   83   89   83   89   95   98  107  100  103  109  115  111\n  118  128  125  135  131  145  141  149  147  159  153  167  180  182\n  186  197  195  203  202  218  220  216  224  228  228  239  248  251\n  257  255  264  266  270  281  277  284  297  299  308  319  336  334\n  332  339  343  352  351  363  359  379  383  394  401  401  407  409\n  411  417  439  437  437  440  464  466  471  465  491  488  497  502\n  519  516  521  529  547  543  546  555  565  572  582  587  587  598\n  599  608  606  616  627  632  636  636  659  657  666  661  672  681\n  687  694  696  722  719  728  737  748  748  763  765  776  780  791\n  787  794  803  808  819  840  839  846  850  848  861  862  861  866\n  878  879  887  919  923  926  934  953  964  978  978  982  9

In [82]:
# convert the input strings to arrays of integers
Eng458_numadmissible = [int(x) for x in Eng458_str_data[0][6][1:-1].split()]
Eng459_numadmissible = [int(x) for x in Eng459_str_data[0][6][1:-1].split()]

In [99]:
# compare these two arrays, looking for the first differences between the (458,3240)-counterexample and the
#  parent (459,3242)-counterexample
i = 0

while Eng458_numadmissible[i] == Eng459_numadmissible[i]:
    i += 1

ip0 = int(Eng458_str_data[0][1])+4  # the index in smallp[] for the array of num_admissible residues
ip1 = ip0+i

print(f"at {i} : p0={smallp[ip0]} p1={smallp[ip1]} {Eng458_numadmissible[i:(i+10)]} vs {Eng459_numadmissible[i:(i+10)]}")
i0 = i

at 16 : p0=137 p1=227 [13, 7, 12, 9, 18, 16, 18, 26, 20, 17] vs [12, 7, 11, 9, 18, 16, 18, 26, 20, 17]


In [119]:

i = i0

while smallp[i+ip0] < 1625:
    if Eng458_numadmissible[i] != Eng459_numadmissible[i]:
        if smallp[i+ip0] < 460:
            print(f"{i:4d} ip {ip0+i:4d} p {smallp[i+ip0]:4d}   nu458 {Eng458_numadmissible[i]:4d}  nu459 {Eng459_numadmissible[i]:4d}")
        else:
            print(f"{i:4d} ip {ip0+i:4d} p {smallp[i+ip0]:4d}   nu458 {Eng458_numadmissible[i]:4d}  nu459 {Eng459_numadmissible[i]:4d}  excess {(Eng459_numadmissible[i]-smallp[i+ip0]+460):4d}")
        
    i += 1
    

  16 ip   48 p  227   nu458   13  nu459   12
  18 ip   50 p  233   nu458   12  nu459   11
  26 ip   58 p  277   nu458   22  nu459   21
  27 ip   59 p  281   nu458   29  nu459   28
  28 ip   60 p  283   nu458   23  nu459   22
  36 ip   68 p  347   nu458   53  nu459   52
  37 ip   69 p  349   nu458   50  nu459   49
  38 ip   70 p  353   nu458   55  nu459   54
  42 ip   74 p  379   nu458   65  nu459   64
  44 ip   76 p  389   nu458   83  nu459   82
  48 ip   80 p  419   nu458   95  nu459   94
  49 ip   81 p  421   nu458   98  nu459   97
  50 ip   82 p  431   nu458  107  nu459  106
  52 ip   84 p  439   nu458  103  nu459  102
  56 ip   88 p  461   nu458  118  nu459  117  excess  116
  60 ip   92 p  487   nu458  131  nu459  130  excess  103
  61 ip   93 p  491   nu458  145  nu459  144  excess  113
  62 ip   94 p  499   nu458  141  nu459  140  excess  101
  68 ip  100 p  547   nu458  180  nu459  179  excess   92
  69 ip  101 p  557   nu458  182  nu459  181  excess   84
  71 ip  103 p  569   

In [126]:
# consolidating the above blocks into a function so that we can get the data across all the (458,3240)-counterexamples
def adm_compare(ic458):
    
    # The indices occur in blocks
    if ic458 < 29:
        ic459 = ic458
    elif ic458 < 58:
        ic459 = ic458-29
    elif ic458 < 87:
        ic459 = ic458-29
    elif ic458 < 116:
        ic459 = ic458-58

    filename = 'Eng458_' + str(ic458) + '_results.csv'
    Eng458_str_data = []
    with open(filename, 'r') as fqtr:
        data_reader = csv.reader(fqtr)
        for row in data_reader:
            Eng458_str_data.append([x for x in row])
    fqtr.close()

    filename = 'Eng459_' + str(ic459) + '_results.csv'
    Eng459_str_data = []
    with open(filename, 'r') as fqtr:
        data_reader = csv.reader(fqtr)
        for row in data_reader:
            Eng459_str_data.append([x for x in row])
    fqtr.close()

    # convert the input strings to arrays of integers
    Eng458_numadmissible = [int(x) for x in Eng458_str_data[0][6][1:-1].split()]
    Eng459_numadmissible = [int(x) for x in Eng459_str_data[0][6][1:-1].split()]

    # Compare these two arrays, looking for the first differences between the (458,3240)-counterexample and the
    #  parent (459,3242)-counterexample
    # We record the number of admissible instances starting at smallp[31]=131, ending at smallp[256]=1621
    
    i0 = 31  # index of p=131 in smallp[]
    i1 = 256 # index of p=1621 in smallp[]

    # prepare the return array
    num_adm = np.ones((2,(i1-i0+1)),dtype=int)

    ip0 = int(Eng458_str_data[0][1])+4  # the index in smallp[] for the array of num_admissible residues

    i = ip0-i0
    if i < 0:
        print(f"Error {i} from {ip0} - {i0}")
        i=0

    # prepare the data for return
    num_adm[0][i:] = Eng458_numadmissible[0:(i1-i0-i+1)]
    num_adm[1][i:] = Eng459_numadmissible[0:(i1-i0-i+1)]

    return(num_adm)

In [125]:
smallp[25:35], smallp[31], smallp[256]

(array([101, 103, 107, 109, 113, 127, 131, 137, 139, 149]),
 np.int64(131),
 np.int64(1621))

In [138]:
ic = 0

all_num_adm = []

while ic < 116:
    num_adm = adm_compare(ic)

    all_num_adm.extend(num_adm)

    ic += 1

with open('Eng458459adm.csv', 'w', newline='\n') as fptr:
    writer = csv.writer(fptr, delimiter=',')
    writer.writerows(all_num_adm)

    

In [133]:
trythis.shape

(2, 226)

In [134]:
trythis[:,0:20]

array([[ 1,  1,  2,  1,  1,  3,  1,  3,  2,  4,  3,  9,  5,  5,  5, 12,
         8, 13,  9, 11],
       [ 1,  1,  2,  1,  1,  3,  1,  3,  2,  4,  3,  9,  5,  5,  5, 12,
         8, 12,  9, 10]])

In [135]:
trythis[:,-20:]

array([[1040, 1047, 1058, 1069, 1078, 1091, 1096, 1098, 1106, 1111, 1116,
        1123, 1127, 1141, 1144, 1149, 1150, 1154, 1161, 1162],
       [1039, 1046, 1057, 1068, 1077, 1091, 1096, 1097, 1105, 1110, 1115,
        1123, 1126, 1140, 1143, 1148, 1150, 1153, 1160, 1162]])

In [121]:
current_s = Eng_consts[23]
rezlist = admissible(727, current_s)
len(rezlist)

336

In [118]:
rezlist

{3,
 4,
 6,
 18,
 20,
 22,
 26,
 27,
 28,
 29,
 38,
 45,
 49,
 50,
 58,
 60,
 62,
 68,
 69,
 70,
 72,
 76,
 87,
 88,
 90,
 93,
 96,
 102,
 105,
 106,
 110,
 112,
 117,
 120,
 130,
 133,
 134,
 136,
 142,
 150,
 156,
 159,
 161,
 162,
 171,
 172,
 176,
 178,
 180,
 182,
 185,
 186,
 192,
 200,
 201,
 205,
 208,
 214,
 221,
 222,
 224,
 235,
 240,
 241,
 245,
 246,
 248,
 256,
 260,
 270,
 275,
 279,
 287,
 291,
 297,
 298,
 306,
 308,
 311,
 316,
 317,
 321,
 324,
 328,
 336,
 339,
 351,
 357,
 358,
 363,
 370,
 374,
 375,
 388,
 390,
 399,
 404,
 405,
 413,
 420,
 425,
 436,
 438}

In [122]:
Eng_consts[23]

array([ 2,  4, 14,  4,  6,  2, 10,  2,  6,  6, 10,  6,  2, 10,  6, 14, 10,
        2,  4, 12,  2,  6,  4,  6,  6,  6,  8,  6,  6,  4,  6,  8,  6, 18,
       16,  6,  6,  8, 10,  2,  4,  8,  6, 12, 10,  2, 10,  6,  2, 10,  8,
        4,  6, 14,  6,  4,  2,  6, 22,  2,  4,  2, 12, 10,  8,  4,  8, 16,
        2,  4,  6,  8,  6,  4, 12,  2, 10,  2, 10,  6,  8,  6, 10,  6,  2,
        6,  4,  8, 10,  8,  4,  2, 10,  8,  6, 10,  2,  4,  6,  8, 10, 14,
       10,  6, 12,  2, 12,  4,  2,  4,  8, 10,  6,  6,  6,  8,  4,  8, 18,
        4,  2,  6,  4,  8, 16,  6,  6,  2,  6, 10,  6, 14,  4,  2,  4,  2,
       24,  6, 12, 10,  2,  4, 12, 14, 16,  8,  4,  6,  2,  4,  8, 16, 14,
       16, 12,  8,  4,  6,  2, 12, 10,  2,  4,  2,  4,  6, 12,  2, 10,  8,
        6,  6, 10,  6,  8,  4,  6,  2, 18, 16,  6,  2,  6,  6, 10,  6,  8,
        4,  2,  4, 12,  2, 10,  2,  4, 12,  2, 10,  2,  4,  6,  8,  6,  6,
        6,  4,  6, 18,  2,  4,  8, 10, 18,  2,  6,  6,  4,  6, 14, 10,  2,
       10, 14,  4,  6,  6

In [104]:
smallp[32], smallp[255]

(np.int64(137), np.int64(1619))

In [39]:
# comparing average gap sizes for small primes, for the counterexample, and for the cycle G(113#)
print(f"pi avg {(3251/458):.4f} vs mu_Eng {(3240/458):.4f} vs mu_113 {(1/mu_recip):.4f}")

pi avg 7.0983 vs mu_Eng 7.0632 vs mu_113 8.7131


In [40]:
mu_gaps = np.zeros(100)
mu_gaps[0] = 2
i=1
while (i<100):
    mu_gaps[i] = mu_gaps[i-1] * (smallp[i]/(smallp[i]-1))
    i += 1

In [41]:
mu_gaps[0:20]

array([2.        , 3.        , 3.75      , 4.375     , 4.8125    ,
       5.21354167, 5.53938802, 5.8471318 , 6.11291052, 6.33122875,
       6.54226971, 6.72399942, 6.89209941, 7.05619701, 7.2095926 ,
       7.34823861, 7.47493238, 7.59951459, 7.71465875, 7.82486816])

In [42]:
smallp[444:470]

array([3121, 3137, 3163, 3167, 3169, 3181, 3187, 3191, 3203, 3209, 3217,
       3221, 3229, 3251, 3253, 3257, 3259, 3271, 3299, 3301, 3307, 3313,
       3319, 3323, 3329, 3331])

In [43]:
smallp[0:5]

array([ 2,  3,  5,  7, 11])

In [44]:
# searching for instances that could survive the sieve.  Far too many copies to search exhaustively, so we pursue a greedy random
# algorithm.  Search randomly for an mk=0, then search greedily for additional 0's
nprimes = len(smallp)
print(f" nprimes {nprimes} maxp {smallp[nprimes-1]}")

 nprimes 646030 maxp 9699691


In [45]:
# random probes through the admissible extensions from the prefixes in gammam_list
# 
k0 = len(gammam_list[0])  # first open index for pk.  Remember the shift pk=smallp[k+4].  smallp[4]=11
k1 = len(rezlist)
print(f"k {k0}-{k1} pmax {smallp[k1+4]}")
logpml = np.log10(2310)
i=5
while (i < (k1+4)):
    logpml = logpml + np.log10(smallp[i])
    i += 1
print(f"i {i-1} p {smallp[i-1]} log(pml) {logpml}")

k 37-665 pmax 5003
i 668 p 4999 log(pml) 2133.1221880361404


## random extensions
The opportunistic random search is in notebook '24_nonconvex_search_Engelsma459'.  
Starting from the prefixes that we have, we search until we find an mk=0, then search for adjacent mk=0


In [50]:
# For the primorial coordinates for the counterexample we start in G(11#), so we start with the unique driving term in that cycle
Engelsma458_11 = np.array([2,4,2,4,6,2,6,4,2,4,6,6,2,6,6,6,4,6,8,4,2,4,2,4,8,6,4,8,4,6,2,6,6,4,2,4,6,8,4,2,4,2,10,2,10,2,4,2,4,6,2,10,2,4,6,8,6,4,2,6,4,6,8,4,6,2,4,
              8,6,4,6,2,4,6,2,6,6,4,6,6,2,6,6,4,2,10,2,10,2,4,2,4,6,2,6,4,2,10,6,2,6,4,2,6,4,6,8,4,2,4,2,12,6,4,6,2,4,6,2,12,4,2,4,8,6,4,2,4,2,10,2,10,6,2,
              4,6,2,6,4,2,4,6,6,2,6,4,2,10,6,8,6,4,2,4,8,6,4,6,2,4,6,2,6,6,6,4,6,2,6,4,2,4,2,10,12,2,4,2,10,2,6,4,2,4,6,6,2,10,2,6,4,14,4,2,4,2,4,8,6,4,
              6,2,4,6,2,6,6,4,2,4,6,2,6,4,2,4,12,2,12,4,2,4,6,2,6,4,2,4,6,6,2,6,4,2,6,4,6,8,4,2,4,2,4,14,4,6,2,10,2,6,6,4,2,4,6,2,10,2,4,2,12,10,2,4,2,
              4,6,2,6,4,6,6,6,2,6,4,2,6,4,6,8,4,2,4,6,8,6,10,2,4,6,2,6,6,4,2,4,6,2,6,4,2,6,10,2,10,2,4,2,4,6,8,4,2,4,12,2,6,4,2,6,4,6,12,2,4,2,4,8,6,4,6,2,
              4,6,2,6,10,2,4,6,2,6,4,2,4,2,10,2,10,2,4,6,6,2,6,6,4,6,6,2,6,4,2,6,4,6,8,4,2,6,4,8,6,4,6,2,4,6,8,6,4,2,10,2,6,4,2,4,2,10,2,10,2,4,2,4,8,6,4,
              2,4,6,6,2,6,4,8,4,6,8,4,2,4,2,4,8,6,4,6,6,6,2,6,6,4,2,4,6,2,6,4,2,4,2,10,2,10,2,6,4,6,2,6,4,2,4,6,6,8,4,2,6,10,8,4,2,4,2,4,8,10,6,2,4,8,6,
              6,4,2,4,6,2,6,4,6,2,10,2,10,2,4,2,4,6,2,6,4,2,4,6,6,2,6,6,6,4,6,8,4,2,4,2,4,8,6,4,8,4,6,2,6,6,4,2,4,6,8,4,2,4,2,10,2,10,2,4,2,4,6,2,10,2,
              4,6,8,6,4,2,6,4,6,8,4,6,2,4,8,6,4,6,2,4,6,2,6,6,4,6,6,2,6,6,4,2,10,2,10,2,4,2,4,6,2,6,4,2,10,6,2,6,4,2,6,4,6,8,4,2,4,2,12,6,4,6,2,4,6,2,
              12,4,2,4,8,6,4,2,4,2,10,2,10,6,2,4,6,2,6,4,2,4,6,6,2,6,4,2,10,6,8,6,4,2,4,8,6,4,6,2,4,6,2,6,6,6,4,6,2,6,4,2,4,2,10,12,2,4,2,10,2,6,4,2,4,6,
              6,2,10,2,6,4,14,4], dtype=int)
print(f"Driving term for Engelsma ({len(Engelsma458_11)},{np.sum(Engelsma458_11)}) counterexample")

Driving term for Engelsma (673,3240) counterexample


In [26]:
prime_constellation[0:459]

array([ 2,  1,  2,  2,  4,  2,  4,  2,  4,  6,  2,  6,  4,  2,  4,  6,  6,
        2,  6,  4,  2,  6,  4,  6,  8,  4,  2,  4,  2,  4, 14,  4,  6,  2,
       10,  2,  6,  6,  4,  6,  6,  2, 10,  2,  4,  2, 12, 12,  4,  2,  4,
        6,  2, 10,  6,  6,  6,  2,  6,  4,  2, 10, 14,  4,  2,  4, 14,  6,
       10,  2,  4,  6,  8,  6,  6,  4,  6,  8,  4,  8, 10,  2, 10,  2,  6,
        4,  6,  8,  4,  2,  4, 12,  8,  4,  8,  4,  6, 12,  2, 18,  6, 10,
        6,  6,  2,  6, 10,  6,  6,  2,  6,  6,  4,  2, 12, 10,  2,  4,  6,
        6,  2, 12,  4,  6,  8, 10,  8, 10,  8,  6,  6,  4,  8,  6,  4,  8,
        4, 14, 10, 12,  2, 10,  2,  4,  2, 10, 14,  4,  2,  4, 14,  4,  2,
        4, 20,  4,  8, 10,  8,  4,  6,  6, 14,  4,  6,  6,  8,  6, 12,  4,
        6,  2, 10,  2,  6, 10,  2, 10,  2,  6, 18,  4,  2,  4,  6,  6,  8,
        6,  6, 22,  2, 10,  8, 10,  6,  6,  8, 12,  4,  6,  6,  2,  6, 12,
       10, 18,  2,  4,  6,  2,  6,  4,  2,  4, 12,  2,  6, 34,  6,  6,  8,
       18, 10, 14,  4,  2

In [27]:
# Calculating the magnitude of p# over the first 200 primes.
i = 1
w=1.0
primpk = np.ones(200, dtype=float)
primpk[0] = 2.0
while (i < 130):
    # w *= (smallp[i]-1)
    primpk[i] = smallp[i] * primpk[i-1]
    print(f"{i:3d} p {smallp[i]:4d} p# {primpk[i]}")
    i +=1
# switch to log-primorial
primpk[i]= np.log10(primpk[i-1]) + np.log10(smallp[i])
print(f"{i:3d} p {smallp[i]:4d} log-p# {primpk[i]}")
i += 1
while (i < 200):
    primpk[i]= primpk[i-1] + np.log10(smallp[i])
    print(f"{i:3d} p {smallp[i]:4d} log-p# {primpk[i]}")
    i += 1
    

  1 p    3 p# 6.0
  2 p    5 p# 30.0
  3 p    7 p# 210.0
  4 p   11 p# 2310.0
  5 p   13 p# 30030.0
  6 p   17 p# 510510.0
  7 p   19 p# 9699690.0
  8 p   23 p# 223092870.0
  9 p   29 p# 6469693230.0
 10 p   31 p# 200560490130.0
 11 p   37 p# 7420738134810.0
 12 p   41 p# 304250263527210.0
 13 p   43 p# 1.308276133167003e+16
 14 p   47 p# 6.148897825884914e+17
 15 p   53 p# 3.2589158477190046e+19
 16 p   59 p# 1.9227603501542128e+21
 17 p   61 p# 1.1728838135940697e+23
 18 p   67 p# 7.858321551080267e+24
 19 p   71 p# 5.57940830126699e+26
 20 p   73 p# 4.072968059924903e+28
 21 p   79 p# 3.2176447673406735e+30
 22 p   83 p# 2.670645156892759e+32
 23 p   89 p# 2.3768741896345556e+34
 24 p   97 p# 2.3055679639455188e+36
 25 p  101 p# 2.328623643584974e+38
 26 p  103 p# 2.398482352892523e+40
 27 p  107 p# 2.5663761175949998e+42
 28 p  109 p# 2.79734996817855e+44
 29 p  113 p# 3.1610054640417614e+46
 30 p  127 p# 4.014476939333037e+48
 31 p  131 p# 5.258964790526278e+50
 32 p  137 p# 7.204

In [28]:
smallp[-1]

np.int64(9699691)

In [27]:
smallp[10]

np.int64(31)

In [28]:
smallp[12]

np.int64(41)

In [34]:
# quick check on the variation across lambda for different J, here J=1,2,3,4,5
i=12
lamJ1=1
lamJ2=1
lamJ3=1
lamJ4=1
lamJ5=1
while smallp[i] < 9699650:
    p = smallp[i]
    lamJ1 *= (p-3)/(p-2)
    lamJ2 *= (p-4)/(p-3)
    lamJ3 *= (p-5)/(p-4)
    lamJ4 *= (p-6)/(p-5)
    lamJ5 *= (p-7)/(p-6)
    i += 1

In [35]:
lamJ1, lamJ2, lamJ3, lamJ4, lamJ5

(np.float64(0.2319987728770889),
 np.float64(0.2306167509091224),
 np.float64(0.22919942164901394),
 np.float64(0.2277448486081579),
 np.float64(0.22625092803546332))

In [38]:
smallp[0:5]

array([ 2,  3,  5,  7, 11])

In [39]:
smallp[440:465]

array([3083, 3089, 3109, 3119, 3121, 3137, 3163, 3167, 3169, 3181, 3187,
       3191, 3203, 3209, 3217, 3221, 3229, 3251, 3253, 3257, 3259, 3271,
       3299, 3301, 3307])

In [40]:
delp = smallp[440:462]-smallp[439:461]

In [41]:
delp

array([ 4,  6, 20, 10,  2, 16, 26,  4,  2, 12,  6,  4, 12,  6,  8,  4,  8,
       22,  2,  4,  2, 12])

In [42]:
s32 = Eng_consts[32]
s32.shape

(459,)

In [43]:
s32[-15:]

array([ 6, 10,  2,  6, 10,  6,  6,  2, 10,  2,  6,  4, 14,  4,  2])

In [47]:
genx = sum(s32[0:449])
genx

np.int64(3186)

In [45]:
genx

np.int64(3192)